In [1]:
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline
import mlflow
import mlflow.sklearn

# Load data
with open('train.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Clean and join ingredients into single string per recipe
df['ingredients_clean'] = df['ingredients'].apply(
    lambda x: ' '.join([i.lower().strip() for i in x])
)

print(df[['cuisine', 'ingredients_clean']].head())
print(f'\nTotal recipes: {len(df)}')
print(f'Cuisines: {df["cuisine"].nunique()}')

       cuisine                                  ingredients_clean
0        greek  romaine lettuce black olives grape tomatoes ga...
1  southern_us  plain flour ground pepper salt tomatoes ground...
2     filipino  eggs pepper salt mayonaise cooking oil green c...
3       indian                     water vegetable oil wheat salt
4       indian  black pepper shallots cornflour cayenne pepper...

Total recipes: 39774
Cuisines: 20


In [2]:
X = df['ingredients_clean']
y = df['cuisine']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build pipeline: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, C=5, random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print(classification_report(y_test, y_pred))

Accuracy: 0.7872

              precision    recall  f1-score   support

   brazilian       0.72      0.62      0.67        93
     british       0.55      0.51      0.53       161
cajun_creole       0.77      0.70      0.73       309
     chinese       0.81      0.86      0.83       535
    filipino       0.73      0.56      0.63       151
      french       0.58      0.63      0.61       529
       greek       0.76      0.66      0.71       235
      indian       0.87      0.91      0.89       601
       irish       0.70      0.49      0.58       133
     italian       0.81      0.88      0.84      1568
    jamaican       0.85      0.75      0.80       105
    japanese       0.78      0.74      0.76       284
      korean       0.82      0.75      0.78       166
     mexican       0.90      0.92      0.91      1288
    moroccan       0.83      0.75      0.79       164
     russian       0.64      0.43      0.51        98
 southern_us       0.73      0.80      0.76       864
     span

In [8]:
import mlflow
import joblib
import os

mlflow.set_tracking_uri("http://34.60.170.139:5000")
mlflow.set_experiment("cuisine-classifier")

# Save model locally first
joblib.dump(pipeline, "cuisine_pipeline.joblib")

with mlflow.start_run(run_name="tfidf-logreg-v1"):
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_features", 5000)
    mlflow.log_param("C", 5)
    mlflow.log_param("max_iter", 1000)
    
    acc = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", acc)
    
    mlflow.log_artifact("cuisine_pipeline.joblib")
    
    run_id = mlflow.active_run().info.run_id
    print(f"Model logged. Accuracy: {acc:.4f}")
    print(f"Run ID: {run_id}")

2026/04/03 11:18:29 INFO mlflow.tracking.fluent: Experiment with name 'cuisine-classifier' does not exist. Creating a new experiment.


Model logged. Accuracy: 0.7872
Run ID: eecd0af993fd4d03bd3b4afc3ec161dc
🏃 View run tfidf-logreg-v1 at: http://34.60.170.139:5000/#/experiments/1/runs/eecd0af993fd4d03bd3b4afc3ec161dc
🧪 View experiment at: http://34.60.170.139:5000/#/experiments/1


In [9]:
from mlflow.tracking import MlflowClient

client = MlflowClient("http://34.60.170.139:5000")

# Register the model
result = client.create_registered_model("cuisine-classifier")
print(f"Registered model: {result.name}")

# Create a version pointing to the artifact
source = f"runs:/{run_id}/cuisine_pipeline.joblib"
version = client.create_model_version(
    name="cuisine-classifier",
    source=source,
    run_id=run_id
)
print(f"Model version: {version.version}")

2026/04/03 11:19:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: cuisine-classifier, version 1


Registered model: cuisine-classifier
Model version: 1


In [10]:
# Test prediction
test_ingredients = ["soy sauce", "ginger", "garlic", "rice", "sesame oil"]
test_input = ' '.join(test_ingredients)
prediction = pipeline.predict([test_input])[0]
proba = pipeline.predict_proba([test_input]).max()

print(f'Ingredients: {test_ingredients}')
print(f'Predicted cuisine: {prediction}')
print(f'Confidence: {proba:.2f}')

Ingredients: ['soy sauce', 'ginger', 'garlic', 'rice', 'sesame oil']
Predicted cuisine: chinese
Confidence: 0.72
